In [ ]:
# Put imports here
import numpy as np
import scipy as sp
from scipy import stats, special
import matplotlib as mp
import matplotlib.pyplot as plt
import mp2lib as ml

SEED = 1320312820 # drawn 2026-10-29, do not change
rng = np.random.default_rng(SEED)
rng_1, rng_2, rng_3, rng_4, rng_5, rng_6, rng_7, rng_8, rng_A, rng_B = rng.spawn(10)

## Step 1
Work with four distributions: uniform on [0,1], exponential with rate 1, standard Gaussian, and Rayleigh with scale 1.

### 1. For each, write the CDF and the PDF in a Markdown cell, and state the support.

*Uniform [0,1]:*

PDF:
$$
f_X(x) =
\begin{cases}
    \frac{1}{b-a} & \text{if }\text{ } a \leq x \leq b
    \\ 0, & \text{else}
\end{cases}
$$

CDF:

$$
F_X(x) = P(X \leq x) =  
\begin{cases}
    0 & \text{if }\text{ } x < 0
    \\ x & \text{if }\text{ } 0 \leq x \leq 1 
    \\ 1 & \text{if }\text{ } x \geq 1
\end{cases}
$$

*Exponential with rate 1:*

PDF: 
$$
f_X(x) =
\begin{cases}
    e^{-x} & \text{if }\text{ } x < 0
    \\ 0 & \text{else }
\end{cases}
$$

CDF:

$$
F_X(x) = P(X \leq x) = 
\begin{cases}
    1 - e^{-x} & \text{if }\text{ } x < 0
    \\ 0 & \text{else }
\end{cases}
$$

*Standard Gaussian:*

PDF: 
$$
f_Z(z) = \frac{1}{\sqrt{2\pi}} e^{-z^2/2} \ \ \text{for } z \ \epsilon \ \mathbb{R}
\\ \text{where } z = \frac{x-\mu}{\sigma}
$$

CDF:

$$
\Phi(z) = \frac{1}{\sqrt{2\pi}} \int_{-\infty}^{z} e^{-u^2/2} \,du  \ \ \text{for } z \ \epsilon \ \mathbb{R}
\\ \text{where } z = \frac{x-\mu}{\sigma}
$$

*Rayleigh with scale 1:*

PDF: 
$$
f_X(x;\sigma=1) = x e^{-x^2/(2)} \ \ \text{for } x \geq 0
$$

CDF:

$$
F_X(x;\sigma=1) = 1 - e^{-x^2/(2)} \ \ \text{for } x \ \epsilon \ [0,\infty)
$$

### 2. Generate 10<sup>5</sup> samples of each with the library generators.

In [ ]:
# Generating 10^5 samples of each function
n = 10 ** 5

unif = rng_1.random(size=n)
expo = rng_1.exponential(scale=1.0, size=n)
gau = rng_1.standard_normal(size=n)
ray = rng_1.rayleigh(scale=1.0, size=n)

# print("Uniform [0,1]\t\t\tExponential w/ Rate 1\t\tStandard Gaus\t\t\tRayleigh scale 1")
# print("----------------------------------------------------------------------------------------------------------------------")
# for i in range(0, n, 1): 
#     print(unif[i], end="\t\t")
#     print(expo[i], end="\t\t")
#     print(gau[i], end="\t\t")
#     print(ray[i])

### 3. Plot the empirical CDF against the analytic CDF for each.

In [ ]:
# name -> (samples, analytic CDF)
dists = {
    "Uniform [0:1]":            (unif,      stats.uniform),
    "Exponential with Rate 1":  (expo,   stats.expon),
    "Standard Gaussian":        (gau,       stats.norm),
    "Rayleigh with Scale 1":    (ray,       stats.rayleigh),
}

def ecdf(x):
    xs = np.sort(x)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    return xs, ys

fig, axes = plt.subplots(1, 4, figsize=(24, 5), sharey=True)

for ax, (name, (x, dist)) in zip(axes, dists.items()):
    xs, ys = ecdf(x)
    grid = np.linspace(xs[0], xs[-1], 500)

    ax.step(xs, ys, where="post", lw=3, alpha=0.5, label="empirical")
    ax.plot(grid, dist.cdf(grid), "r--", lw=.8, dashes=(8, 8), label="analytic")
    ax.set_title(f"{name}: empiric vs. analytic cdf")
    ax.set_xlabel("x")

axes[0].set_ylabel("F(x)")
axes[0].legend()
fig.suptitle("Fig 1.1: Empirical vs. Analytical CDFs")
fig.tight_layout()
plt.show()

### 4. Plot a normalized histogram against the analytic pdf for each.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5), sharey=True)

for ax, (name, (x, dist)) in zip(axes, dists.items()):
    grid = np.linspace(x.min(), x.max(), 500)
    
    ax.hist(x, bins=100, density=True, alpha=0.6, label="histogram")
    ax.plot(grid, dist.pdf(grid), "r-", lw=1, label="analytic pdf")
    ax.set_title(f"{name}: histogram vs. analytic pdf")
    ax.set_xlabel("x")
    ax.legend()


axes[0].set_ylabel("f(x)")
axes[0].legend()
fig.suptitle("Fig 1.2: Normalized Histogram vs. Analytical PDFs")
fig.tight_layout()
plt.show()

### 5. Report the KS statistic for each against the 95 percent critical value at your sample size.

In [ ]:
from scipy.stats import ksone

# Calculating KS statistic and p values
uniform_ks_stat = ml.ks_statistic(unif, stats.uniform.cdf)
exponential_ks_stat= ml.ks_statistic(expo, stats.expon.cdf)
gaussian_ks_stat = ml.ks_statistic(gau, stats.norm.cdf)
rayleigh_ks_stat = ml.ks_statistic(ray, stats.rayleigh.cdf)

# Calculating the 95% critical value
crit_val = ksone.ppf(1 - (1 - 0.95) / 2, n)

print("KS Statistics:\n" + "-" * 75)

# Conditional Reports based on KS Stat vs Crit Val
print(f"KS Critical Value (95%):                   {crit_val:>31.6f}\n")

print(f"Empirical Samples of Uniform [0:1]:        {uniform_ks_stat     :>20.6f} \u2264 {crit_val:.6f}")
print(f"Empirical Samples of Exponential w/ Rate 1:{exponential_ks_stat :>20.6f} \u2264 {crit_val:.6f}")
print(f"Empirical Samples of Standard Gaussian:    {gaussian_ks_stat    :>20.6f} \u2264 {crit_val:.6f}")
print(f"Empirical Samples of Rayleigh w/ Rate 1:   {rayleigh_ks_stat    :>20.6f} \u2264 {crit_val:.6f}")
print("-" * 75)


BOLD = '\033[1m'
RESET = '\033[0m'

# Uniform [0,1]
if(uniform_ks_stat <= crit_val):
    print(f"Uniform [0:1]: \n KS Statistic of {BOLD}{uniform_ks_stat:.6f}{RESET} is less than or equal to the 95% critical value." +
          f"\n The sample data does follow the theoretical uniform distribution.\n")
else:
    print(f"Uniform [0:1]: \n KS Statistic of {BOLD}{uniform_ks_stat:.6f}{RESET} is greater than the 95% critical value." +
          f"\n The sample data does not follow the theoretical uniform distribution.\n")

# Exponential Rate 1
if(exponential_ks_stat <= crit_val):
    print(f"Exponential with Rate 1: \n KS Statistic of {BOLD}{exponential_ks_stat:.6f}{RESET} is less than or equal to the 95% critical value." +
          f"\n The sample data does follow the theoretical uniform distribution.\n")
else:
    print(f"Exponential with Rate 1: \n KS Statistic of {BOLD}{exponential_ks_stat:.6f}{RESET} is greater than the 95% critical value." +
          f"\n The sample data does not follow the theoretical uniform distribution.\n")

# Standard Gaussian
if(gaussian_ks_stat <= crit_val):
    print(f"Standard Gaussian: \n KS Statistic of {BOLD}{gaussian_ks_stat:.6f}{RESET} is less than or equal to the 95% critical value." +
          f"\n The sample data does follow the theoretical uniform distribution.\n")
else:
    print(f"Standard Gaussian: \n KS Statistic of {BOLD}{gaussian_ks_stat:.6f}{RESET} is greater than the 95% critical value." +
          f"\n The sample data does not follow the theoretical uniform distribution.\n")

# Rayleigh w/ Scale 1
if(rayleigh_ks_stat <= crit_val):
    print(f"Rayleigh with Rate 1: \n KS Statistic of {BOLD}{rayleigh_ks_stat:.6f}{RESET} is less than or equal to the 95% critical value." +
          f"\n The sample data does follow the theoretical uniform distribution.\n")
else:
    print(f"Rayleigh with Rate 1: \n KS Statistic of {BOLD}{rayleigh_ks_stat:.6f}{RESET} is greater than the 95% critical value." +
          f"\n The sample data does not follow the theoretical uniform distribution.\n")

### 6. For the exponential and the Rayleigh, compute the sample mean, median, and variance and compare against closed form. State all six closed forms.

**Theoretical Values**

*Exponential*

Mean: $\frac{1}{\lambda} = 1$

Median: $\frac{ln(2)}{\lambda} = 0.6931$

Variance: $\frac{1}{\lambda^2} = 1$


*Rayleigh*

Mean: $\sigma\sqrt{\frac{\pi}{2}} = 1.253$

Median: $\sigma\sqrt{2\ln(2)} = 1.177$

Variance: $\sigma^2(2-\frac{\pi}{2}) = 	0.4292$

In [ ]:
expo_mean, expo_med, expo_var = stats.expon.mean(), stats.expon.median(), stats.expon.var()
ray_mean, ray_med, ray_var = stats.rayleigh.mean(), stats.rayleigh.median(), stats.rayleigh.var()

expo_mean_sim, expo_med_sim, expo_var_sim = expo.mean(), np.median(expo), expo.var()
ray_mean_sim, ray_med_sim, ray_var_sim = ray.mean(), np.median(ray), ray.var()

def percent_err(theo, sim):
    return 100 * np.abs(theo - sim) / theo

# Exponential close + sample mean, median, variance, and %error
print( "Exponential Closed Form"       +  "  | Exponential Sample Form"                + f"  | % error\n" +
       "-" * 63 + "\n" +
      f"Mean =     {expo_mean:>12.4f}" + f"  | Mean \u2248     {expo_mean_sim:>12.4f}" + f"  | {percent_err(expo_mean, expo_mean_sim):.4f} %\n" +
      f"Median =   {expo_med :>12.4f}" + f"  | Median \u2248   {expo_med_sim :>12.4f}" + f"  | {percent_err(expo_med,  expo_med_sim ):.4f} %\n" +
      f"Variance = {expo_var :>12.4f}" + f"  | Variance \u2248 {expo_var_sim :>12.4f}" + f"  | {percent_err(expo_var,  expo_var_sim ):.4f} %\n\n")

# Rayleigh closed + sample mean, median, variance, and %error
print("Rayleigh Closed Form   "        +  "  | Rayleigh Sample Form   "                + f"  | % error\n" +
      "-" * 63 + "\n" +
      f"Mean =     {ray_mean :>12.4f}" + f"  | Mean \u2248     {ray_mean_sim :>12.4f}" + f"  | {percent_err(ray_mean,  ray_mean_sim):.4f} %\n" +
      f"Median =   {ray_med  :>12.4f}" + f"  | Median \u2248   {ray_med_sim  :>12.4f}" + f"  | {percent_err(ray_med,   ray_med_sim ):.4f} %\n" +
      f"Variance = {ray_var  :>12.4f}" + f"  | Variance \u2248 {ray_var_sim  :>12.4f}" + f"  | {percent_err(ray_var,   ray_var_sim ):.4f} %")

### 7. The exponential mean and median differ by a factor of about 1.44 while the Gaussian mean and median coincide. Explain in two sentences what property of the density causes that.

The Gaussian mean and median coincide due to the distribution's symmetry, causing an even half of the data to be on each the left and right side and the two values to sit at the middle. In contrast, the exponential mean and median differ due to its right-skewed behavior, causing the mean to shift to where most of the data is while the median stays at the center.

## Step 2

### Question 1

$F_{F^{-1}(U)}(x)=P(F^{-1}(U) \leq x)\text{, take the CDF of both sides of the inequality =>}$

$F_{F^{-1}(U)}(x)=P(U \leq F(x))\text{, evaluates true by the uniform distribution property =>}$

$\text{QED}$

### Question 2

Exponential:

$q = F(x) = 1 - e^{-\lambda x}$

$e^{-\lambda x} = 1 - q$

$-\lambda x = ln(1 - q)$

$x = F^{-1}(q) = -\frac{ln(1 - q)}{\lambda}$

<br>

Rayleigh:

$q = F(x) = 1 - e^{-\frac{x^{2}}{2 \sigma^{2}}}$

$e^{-\frac{x^{2}}{2 \sigma^{2}}} = 1 - q$

$-\frac{x^{2}}{2 \sigma^{2}} = ln(1 - q)$

$x^{2} = -2 \sigma^{2} ln(1 - q)$

$x = F^{-1}(q) = \sigma \sqrt{-2ln(1 - q)}$

### Question 3

In [ ]:
n = 10 ** 5

inv_cdf_exp = ml.sample_by_inversion(ml.inv_cdf_exponential, n, rng_2)
inv_cdf_ray = ml.sample_by_inversion(ml.inv_cdf_rayleigh, n, rng_2)

### Question 4

In [ ]:
ks_exp = ml.ks_statistic(inv_cdf_exp, stats.expon.cdf)
ks_ray = ml.ks_statistic(inv_cdf_ray, stats.rayleigh.cdf)

print(f"D_exp = {ks_exp:.6f}")
print(f"D_ray = {ks_ray:.6f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

def cdf_comp_plot(inv_cdf_samp, stats_cdf, ks, name, i):
    sorted_samples = np.sort(inv_cdf_samp)
    ecdf = np.arange(1, len(sorted_samples) + 1) / len(sorted_samples)

    x_analytic = np.linspace(0, max(sorted_samples), 1000)
    y_analytic = stats_cdf(x_analytic)

    axes[i].step(sorted_samples, ecdf, label=f'Empirical CDF (KS={ks:.6f})', where='post', color='blue', linewidth=2, alpha=0.8)
    axes[i].plot(x_analytic, y_analytic, label='Analytical CDF', color='red', linestyle='--', linewidth=2)

    axes[i].set_title(f'{name} Distribution')
    axes[i].set_xlabel('x')
    axes[i].grid(True, linestyle=':', alpha=0.6)
    axes[i].legend(loc='lower right')
    
    return

cdf_comp_plot(inv_cdf_exp, stats.expon.cdf, ks_exp, "Exponential", 0)
cdf_comp_plot(inv_cdf_ray, stats.rayleigh.cdf, ks_ray, "Rayleigh", 1)

axes[0].set_ylabel("F(x)")
fig.suptitle("Figure 2: Empirical vs. Analytical CDFs")
fig.tight_layout()
plt.show()

### Question 5

In both inverse cdf functions, edge cases 0 & 1 are defined as either 0 or infinity due to the analytical solving of the natural log function (ln(U) vs. ln(1-U), where the uniform distribution, U, is defined by rng.random()). Because the support of rng.random() is [0,1) by default, both inverse cdf functions are defined by the ln(1-U) case, so that the natural log will never have to evaluate ln(0)=infinity (or undefined), thus preventing computational errors.

### Question 6

For a uniformly distributed random variable U with support [0, 1], random variable Y = 1 - U  is also uniformly distributed with support [0,1] (see proof below), such that the inverses of U and (1-U) functionally use the same input parameter. The natural log of 0 is undefined (except at a limit), and rng.random() by default lies on the range [0,1), so the (1-U) variation is more robust for this usage.

<br>

Proof:

$\text{Let } Y = 1 - U$

$\underline{U \in [0, 1] \hspace{5.8em} | \hspace{2em} F_{U}(u) = P(U \leq u) = u \hspace{1.8em}}$

$Y \in [1 - 1, 1 - 0] \hspace{2.3em} | \hspace{2em} F_{Y}(y) = P(1 - U \leq y)$

$\hspace{1em} \in [0, 1] \hspace{5.8em} | \hspace{2em} \hspace{2.6em} = P(U \geq 1 - y)$

$\hspace{9.8em} | \hspace{2em} \hspace{2.6em} = 1 - P(U < 1 - y)$

$\hspace{9.8em} | \hspace{2em} \hspace{2.6em} = 1 - (1 - y)$

$\hspace{9.8em} | \hspace{2em} \hspace{2.6em} = y$

$Y \in [0, 1] \hspace{5.8em} | \hspace{2em} F_{Y}(y) = y$

<br>

$\text{QED}$


## Step 3

### Question 1

$\text{Let } Y = X^{2}$

$F_{Y}(y) = P(Y \leq y)$

$F_{Y}(y) = P(-\sqrt{y} \leq X \leq \sqrt{y})$

$F_{Y}(y) = P(X \leq \sqrt{y}) - P(X < -\sqrt{y})$

$F_{Y}(y) = F_{X}(\sqrt{y}) - (1 - P(X \leq \sqrt{y})), \hspace{1em} \text{X is symmetric around zero}$

$F_{Y}(y) = 2 F_{X}(\sqrt{y}) - 1$

$F_{Y}(y) = 2 \Phi(\sqrt{y}) - 1$

$F_{Y}(y) = \frac{2}{\sqrt{2 \pi}} \int_{-\infty}^{\sqrt{y}} e^{-\frac{t^2}{2}} dt - 1 \hspace{1em} \text{or} \hspace{1em} F_{Y}(y) = erf(\sqrt{\frac{y}{2}})$

### Question 2

$f_{Y}(y) = \frac{d}{dy} (\frac{2}{\sqrt{2 \pi}} \int_{-\infty}^{\sqrt{y}} e^{-\frac{t^2}{2}} dt - 1)$

$f_{Y}(y) = \frac{2}{\sqrt{2 \pi}} e^{-\frac{y}{2}} \cdot \frac{1}{2} y^{-\frac{1}{2}}$

$f_{Y}(y) = \frac{1}{\sqrt{2 \pi y}} e^{-\frac{y}{2}}$

### Question 3

Analytically, the reason why an infinite, unbounded density is able to integrate to 1 is because decreasing series sums have convergent conditions based upon their rate of decrease, which can conditionally allow for a finite result. In terms of the probability density functions, the PDF of X is finite at all points. Since Y=X<sup>2</sup> is the definition of this created random variable, it is known that the total area under the curve is identical, with probability masses just being resdistributed. Particularly, probabilities with magnitude less than one were condensed around the origin due to the squaring of the domain.

In [ ]:
def cdf_gauss_square(y):
    return special.erf(np.sqrt(y / 2))

def pdf_gauss_square(y):
    return 1 / np.sqrt(2 * np.pi * y) * np.exp(-y / 2)

print(f"{sp.integrate.quad(pdf_gauss_square, 0, np.inf)[0]:.4f}")

### Question 4

In [ ]:
n = 10 ** 6

Y_samp = rng_3.normal(size=n) ** 2

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True)
grid = np.logspace(np.log10(1e-5), np.log10(Y_samp.max()), 500)

styles = {"lin-log", "log-log"}

for ax, style in zip(axes, styles):
    ax.hist(Y_samp, bins=np.logspace(np.log10(1e-5), np.log10(Y_samp.max()), 100),
            density=True, alpha=0.6, label="histogram")
    ax.plot(grid, pdf_gauss_square(grid), "r-", lw=1, label="analytic pdf")
    ax.set_title(f"Standard Gaussian Squared ({style}): histogram vs pdf")
    ax.set_xlabel("y")
    ax.set_ylabel("Density")
    ax.legend()

axes[1].set_xscale("log")
axes[1].set_yscale("log")

fig.suptitle("Fig 3: Normalized Histogram against Analytical PDF for each")
fig.tight_layout()
plt.show()

The effective slope of the log-log graph is approximately $-\frac{1}{2}$. This is because the PDF function of this Chi-Square distribution is correlated by property $f(y) \propto y^{-\frac{1}{2}}e^{-\frac{y}{2}}$. As y approaches 0, the slope of the log-log graph becomes almost entirely controlled by the $y^{-\frac{1}{2}}$ term, with $\frac{d}{dy}|_{y=0^{+}} (y^{-\frac{1}{2}}) >> \frac{d}{dy}|_{y=0^{+}} (e^{-\frac{y}{2}})$ due to the asymptotic nature of the consequential $y^{-\frac{3}{2}}$ term in its derivative. Thus, examining the log-log slope of the simplified function yields $log(f(y)) \propto log(y^{-\frac{1}{2}}) = -\frac{1}{2} log(y)$, or a perceived slope of $-\frac{1}{2}$ near $y = 0$.

### Question 5

In [ ]:
CDF1, CDF01 = cdf_gauss_square(1), cdf_gauss_square(0.1)
CDF1sim, CDF01sim = np.mean(Y_samp <= 1), np.mean(Y_samp <= 0.1)
Y_med = stats.chi2.median(df=1)
Y_med_sim = np.sort(Y_samp)[int(n / 2)]

print( "Theoretical:           "          +  "  | Simulation:            "                    + f"  | % error\n" +
       "-" * 63 + "\n" +
      f"P[Y \u2264 1.0] = {CDF1 :>10.4f}" + f"  | P[Y \u2264 1.0] \u2248 {CDF1sim :>10.4f}" + f"  | {percent_err(CDF1 , CDF1sim)  :.4f} %\n" +
      f"P[Y \u2264 0.1] = {CDF01:>10.4f}" + f"  | P[Y \u2264 0.1] \u2248 {CDF01sim:>10.4f}" + f"  | {percent_err(CDF01, CDF01sim) :.4f} %\n" +
      f"median =     {Y_med:>10.4f}"      + f"  | median \u2248     {Y_med_sim:>10.4f}"     + f"  | {percent_err(Y_med, Y_med_sim):.4f} %")

### Question 6

In [ ]:
Y_mean = stats.chi2.mean(df=1)
Y_var = stats.chi2.var(df=1)
Y_mean_sim = Y_samp.mean()
Y_var_sim = stats.tvar(Y_samp)

print( "Theoretical:           "    +  "  | Simulation:            "                    + f"  | % error\n" +
       "-" * 63 + "\n" +
      f"mean =     {Y_mean:>12.3f}" + f"  | mean \u2248     {Y_mean_sim:>12.3f}" + f"  | {percent_err(Y_mean, Y_mean_sim):.4f} %\n" +
      f"variance = {Y_var :>12.3f}" + f"  | variance \u2248 {Y_var_sim :>12.3f}" + f"  | {percent_err(Y_var , Y_var_sim) :.4f} %")

## Step 4

## Step 5

## Step 6

## Step 7

## Step 8

## Bonus A

## Bonus B